# Restaurant Tips Analysis & High Tip Prediction

Main Colab notebook: https://colab.research.google.com/drive/1PPFQPXRzfr8JUdseor-0yq6nQ13NscJ5?usp=sharing

This GitHub notebook is a clean portfolio copy of the Colab project.

In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', None)
DATA_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
OUTPUT_DIR = Path('output')

## Load dataset from GitHub

In [ ]:
def create_fallback_dataset(seed=93, n=120):
    rng = np.random.default_rng(seed)
    days = ['Thur', 'Fri', 'Sat', 'Sun']
    sexes = ['Female', 'Male']
    smokers = ['Yes', 'No']
    rows = []
    for _ in range(n):
        day = rng.choice(days, p=[0.25, 0.12, 0.33, 0.30])
        time = 'Lunch' if day == 'Thur' and rng.random() < 0.75 else 'Dinner'
        party_size = int(rng.choice([1, 2, 2, 2, 3, 4, 4, 5, 6]))
        total_bill = round(rng.gamma(shape=4.5, scale=4.2) + party_size * 2.5, 2)
        tip_percentage = rng.normal(loc=0.16, scale=0.035)
        if time == 'Dinner':
            tip_percentage += 0.01
        if party_size >= 4:
            tip_percentage -= 0.01
        tip_percentage = max(0.05, min(tip_percentage, 0.32))
        tip = round(total_bill * tip_percentage, 2)
        rows.append({'total_bill': total_bill, 'tip': tip, 'sex': rng.choice(sexes), 'smoker': rng.choice(smokers, p=[0.38, 0.62]), 'day': day, 'time': time, 'size': party_size})
    return pd.DataFrame(rows)

try:
    df = pd.read_csv(DATA_URL)
    data_source = 'GitHub raw CSV'
except Exception:
    df = create_fallback_dataset()
    data_source = 'generated fallback dataset'

print('Data source:', data_source)
print('Dataset shape:', df.shape)
df.head()

## Clean data and create features

In [ ]:
df_clean = df.copy().drop_duplicates()
numeric_cols = ['total_bill', 'tip', 'size']
text_cols = ['sex', 'smoker', 'day', 'time']

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

for col in text_cols:
    df_clean[col] = df_clean[col].astype('string').str.strip().str.title()

df_clean = df_clean.dropna(subset=numeric_cols + text_cols)
df_clean = df_clean[(df_clean['total_bill'] > 0) & (df_clean['tip'] > 0) & (df_clean['size'] > 0)]

df_clean['tip_percentage'] = (df_clean['tip'] / df_clean['total_bill']) * 100
df_clean['bill_per_person'] = df_clean['total_bill'] / df_clean['size']
df_clean['tip_per_person'] = df_clean['tip'] / df_clean['size']

high_tip_threshold = np.percentile(df_clean['tip_percentage'], 75)
df_clean['high_tip'] = np.where(df_clean['tip_percentage'] >= high_tip_threshold, 1, 0)

print('Cleaned shape:', df_clean.shape)
print('High-tip threshold:', round(high_tip_threshold, 2))
df_clean.head()

## KPI summary and analysis

In [ ]:
kpi_summary = pd.DataFrame({
    'Metric': ['Total Records', 'Average Total Bill', 'Average Tip', 'Average Tip Percentage', 'Average Party Size'],
    'Value': [len(df_clean), round(df_clean['total_bill'].mean(), 2), round(df_clean['tip'].mean(), 2), round(df_clean['tip_percentage'].mean(), 2), round(df_clean['size'].mean(), 2)]
})

avg_tip_by_day = df_clean.groupby('day', as_index=False).agg(avg_tip_percentage=('tip_percentage', 'mean')).sort_values('avg_tip_percentage', ascending=False)
analysis_by_time = df_clean.groupby('time', as_index=False).agg(records=('total_bill', 'count'), avg_bill=('total_bill', 'mean'), avg_tip=('tip', 'mean'), avg_tip_percentage=('tip_percentage', 'mean')).sort_values('avg_tip_percentage', ascending=False)
analysis_by_smoker = df_clean.groupby('smoker', as_index=False).agg(records=('total_bill', 'count'), avg_bill=('total_bill', 'mean'), avg_tip=('tip', 'mean'), avg_tip_percentage=('tip_percentage', 'mean')).sort_values('avg_tip_percentage', ascending=False)
avg_bill_by_size = df_clean.groupby('size', as_index=False).agg(avg_total_bill=('total_bill', 'mean')).sort_values('size')

display(kpi_summary)
display(avg_tip_by_day)
display(analysis_by_time)
display(analysis_by_smoker)
display(avg_bill_by_size)

## Visualizations

In [ ]:
avg_tip_by_day.plot(kind='bar', x='day', y='avg_tip_percentage', legend=False, figsize=(8, 5))
plt.title('Average Tip Percentage by Day')
plt.xlabel('Day')
plt.ylabel('Average Tip Percentage')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(df_clean['total_bill'], df_clean['tip'])
plt.title('Total Bill vs Tip')
plt.xlabel('Total Bill')
plt.ylabel('Tip')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(df_clean['tip_percentage'], bins=15)
plt.title('Tip Percentage Distribution')
plt.xlabel('Tip Percentage')
plt.ylabel('Number of Records')
plt.tight_layout()
plt.show()

## Machine learning model

In [ ]:
features = ['total_bill', 'size', 'bill_per_person', 'sex', 'smoker', 'day', 'time']
target = 'high_tip'

X = df_clean[features]
y = df_clean[target]

numeric_features = ['total_bill', 'size', 'bill_per_person']
categorical_features = ['sex', 'smoker', 'day', 'time']

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print('Model accuracy:', round(accuracy, 4))
print('\nClassification report:')
print(classification_report(y_test, predictions))

ConfusionMatrixDisplay.from_predictions(y_test, predictions)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## Validation, export, and final insights

In [ ]:
required_columns = ['total_bill', 'tip', 'size', 'tip_percentage', 'bill_per_person', 'tip_per_person', 'high_tip']
assert len(df_clean) > 0, 'The cleaned dataset should not be empty.'
assert df_clean[required_columns].isna().sum().sum() == 0, 'Required columns should not have missing values.'
assert set(df_clean['high_tip'].unique()).issubset({0, 1}), 'high_tip must only contain 0 and 1.'
assert 0 <= accuracy <= 1, 'Accuracy should be between 0 and 1.'

OUTPUT_DIR.mkdir(exist_ok=True)
df_clean.to_csv(OUTPUT_DIR / 'restaurant_tips_cleaned.csv', index=False)
pd.DataFrame({'metric': ['data_source', 'clean_rows', 'high_tip_threshold', 'model_accuracy'], 'value': [data_source, len(df_clean), round(high_tip_threshold, 2), round(accuracy, 4)]}).to_csv(OUTPUT_DIR / 'model_metrics.csv', index=False)

print('Project validation passed.')
print('Exported files:')
for file in OUTPUT_DIR.iterdir():
    print('-', file)

best_day = avg_tip_by_day.iloc[0]
best_time = analysis_by_time.iloc[0]
best_smoker_group = analysis_by_smoker.iloc[0]

print('\nFinal Business Insights')
print('-----------------------')
print(f"1. The day with the highest average tip percentage is {best_day['day']} with {best_day['avg_tip_percentage']:.2f}%.")
print(f"2. The time with the highest average tip percentage is {best_time['time']} with {best_time['avg_tip_percentage']:.2f}%.")
print(f"3. The smoker group with the highest average tip percentage is {best_smoker_group['smoker']} with {best_smoker_group['avg_tip_percentage']:.2f}%.")
print(f"4. A high-tip order starts at {high_tip_threshold:.2f}% tip percentage.")
print(f"5. The model accuracy is {accuracy:.2%}.")